# 04 · State and memory

A customer asks where order A1003 is, then asks "when will it arrive exactly?" without repeating the
number. In notebooks 01 to 03 every case used a fresh agent on purpose. Here we use the opposite: a
Strands `Agent` **keeps its conversation** in `agent.messages`, so calling the same agent again continues
it.

**You will learn**

- why a fresh agent forgets, and the same agent remembers
- how one agent per customer gives per-conversation memory
- how a session manager saves a conversation to disk and restores it in a new agent
- what changes when memory has to survive a restart

In [1]:
# Imports and setup shared by the cells below.
from strands import Agent

from agent_contracts.harness import verify_model
from agent_contracts.shipfast import tools as shipfast  # the shared ShipFast tools, which also log every call and refund
from strands_agents_impl.llm import make_model
from strands_agents_impl.prompts import SPECIALIST
from strands_agents_impl.tools import shipfast_tools

verify_model()  # stops with the fix if Ollama or the shared model is missing
model, tools = make_model(), shipfast_tools()

# Two messages from one customer: the second only makes sense if the agent remembers the first.
FIRST, FOLLOW_UP = "Hi, where is my order A1003?", "When will it arrive exactly?"


def new_agent(**kwargs) -> Agent:
    """A ShipFast support agent. Extra keyword arguments (such as a session manager) go straight to Agent."""
    return Agent(model=model, tools=tools, system_prompt=SPECIALIST, callback_handler=None, **kwargs)  # None: do not stream to the screen

## Without memory

A fresh agent for the follow-up knows nothing about the first message. On its own, the follow-up has no
order ID, so the right move is to ask which order.

In [2]:
# A fresh agent answers the first message. We ignore the answer; what matters is whether another fresh
# agent knows about it.
new_agent()(FIRST)

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'Your order A1003 is currently delayed and is expected to be delivered by LBC on October 24, 2026. The latest update shows it is held at the Davao hub due to a backlog. We apologize for the inconvenience and are working to resolve this as soon as possible.'}], 'metadata': {'usage': {'inputTokens': 756, 'outputTokens': 66, 'totalTokens': 822}, 'metrics': {'latencyMs': 13186, 'timeToFirstByteMs': 6}}, 'tracking_id': '76c12ce4-93ea-43f1-a3c8-523299485108'}, metrics=EventLoopMetrics(cycle_count=2, tool_metrics={'track_shipment': ToolMetrics(tool={'toolUseId': 'tooluse_c36b2b7b38c04fa9bc6c3fe3', 'name': 'track_shipment', 'input': {'order_id': 'A1003'}}, call_count=1, success_count=1, error_count=0, total_time=0.0004079580103280023)}, cycle_durations=[4.599715948104858, 13.198522090911865], agent_invocations=[AgentInvocation(cycles=[EventLoopCycleMetric(event_loop_cycle_id='af0e69a8-e703-4fb9-9d90-2e7db0d8

Now send the follow-up to a second fresh agent. It shares nothing with the first, so it sees only the follow-up.

In [3]:
shipfast.reset()  # clear the tool-call log, so only this run shows up
reply = new_agent()(FOLLOW_UP)

print("tools:", ", ".join(shipfast.tool_calls()) or "none")
print("reply:", str(reply).strip())  # str() of the result is the agent's answer

tools: track_shipment
reply: I couldn't find any order with the ID "order_12345". Could you please provide the correct order ID so I can check the delivery details for you?


If the tools line shows a lookup, the agent tried an order ID that nothing in this run gave it: a
guess. The shared tools answer `order_not_found` for a made-up ID, which is what turns a guess into a
question to the customer rather than an answer about the wrong order.

## The same agent remembers

Call the **same** agent twice and the second call sees the first. One agent per customer conversation is
the whole mechanism.

In [4]:
shipfast.reset()
maria = new_agent()  # one agent for the whole conversation
for text in (FIRST, FOLLOW_UP):
    reply = maria(text)  # each call appends to maria.messages, and the next call sees all of it
    print(f"customer: {text}\nagent:    {str(reply).strip()[:240]}\n")
print("tools called:", ", ".join(shipfast.tool_calls()) or "none")
print("messages in Maria's conversation:", len(maria.messages))

customer: Hi, where is my order A1003?
agent:    Your order A1003 is currently delayed and is expected to be delivered by LBC on October 24, 2026. The latest update shows it is held at the Davao hub due to a backlog. We apologize for the inconvenience and are working to resolve this as so



customer: When will it arrive exactly?
agent:    Your order A1003 is expected to arrive by LBC on October 24, 2026. The exact delivery date may vary slightly depending on the backlog at the Davao hub. We apologize for the delay and are working to resolve this as quickly as possible.

tools called: track_shipment, track_shipment
messages in Maria's conversation: 8


`agent.messages` is the conversation: user messages, the model's replies and tool calls, and the tool
results. You can read it, trim it, or start an agent with a list of earlier messages.

## Memory that changes the outcome

Scenario S4 needs two turns. The vase from A1004 arrived broken; policy P3 requires a photo before a
refund. The first turn should ask for it, and the second, which never mentions the order, should refund
the $60 total. Only memory connects the two.

In [5]:
shipfast.reset()
s4 = new_agent()  # a new agent is a new conversation
for text in ("My vase set from order A1004 arrived broken.", "Here is the photo: photo_A1004.jpg"):
    reply = s4(text)
    refunds = [f"{r['order_id']} ${r['amount']}" for r in shipfast.refunds()]  # refunds actually issued so far
    print(f"customer: {text}")
    print(f"agent:    {str(reply).strip()[:220]}")
    print(f"ledger:   {', '.join(refunds) or 'no refund'}\n")

customer: My vase set from order A1004 arrived broken.
agent:    The vase set from your order A1004 arrived broken. According to our policy, you must provide a photo of the damage within 7 days of the delivery date. Since the delivery was on October 30, 2026, and today is November 1, 
ledger:   no refund



customer: Here is the photo: photo_A1004.jpg
agent:    Your full refund of $60.00 for the broken vase set in order A1004 has been processed. You should see the refund reflected in your account within a few business days. If you have any further questions, feel free to ask!
ledger:   A1004 $60.0



When we ran it, this is exactly what policy P3 wants: the first turn asked for the photo and refunded
nothing, and the second turn, which never names the order, refunded the $60 total once. (In the CrewAI
tutorial the same two turns refunded on the first turn, before the photo; same model, different framework,
different mistake.)

## Surviving a restart: sessions

`agent.messages` lives in this Python process. A **session manager** saves the conversation as it
happens and restores it into a new agent with the same session id. `FileSessionManager` writes it to a
folder; here a temporary one, so every run starts clean.

In [6]:
import tempfile

from strands.session.file_session_manager import FileSessionManager

folder = tempfile.mkdtemp()  # a new empty folder for every run

# The first agent saves its conversation to the folder under the session id "maria".
shipfast.reset()
first = new_agent(session_manager=FileSessionManager(session_id="maria", storage_dir=folder))
first(FIRST)
del first  # as if the process had ended

Now build a new agent with the same session id and the same folder. It never saw the first message, so anything it knows came from disk.

In [7]:
restored = new_agent(session_manager=FileSessionManager(session_id="maria", storage_dir=folder))

print("messages restored:", len(restored.messages))  # loaded from the folder when the agent was created
print("follow-up reply:  ", str(restored(FOLLOW_UP)).strip()[:240])

messages restored: 4


follow-up reply:   Your order A1003 is expected to arrive by LBC on October 24, 2026. The exact delivery date may vary slightly depending on the backlog at the Davao hub. We apologize for the delay and are working to resolve this as quickly as possible.


The second agent is a new object, built after the first was gone, and it still answers about A1003:
the session manager loaded the conversation from disk. Strands also has session managers for S3 and for
your own storage.

Two limits to keep in mind. Every message is sent back to the model on every turn, so long conversations
grow the prompt until they hit the 16K context; Strands' **conversation manager** (a sliding window by
default) trims older turns. And a conversation is per customer: facts that should follow a customer into
a new conversation need storage of their own.

## How the others do this

- **LangGraph** gives an agent a **checkpointer** and a thread id: every step's state is saved, and calling
  again with the same id continues the conversation. Database checkpointers make it survive restarts.
- **CrewAI** has no per-agent conversation: you keep the history yourself and pass it to `kickoff`, or keep
  it in a Flow's persisted state. Its `Memory` is a separate store of facts searched by meaning.

## Takeaways

- A Strands agent remembers its own conversation; a fresh agent starts empty.
- One agent per customer is per-conversation memory; `agent.messages` is the record.
- A session manager saves the conversation and restores it into a new agent, across restarts.
- Next: **05 · Orchestration**.